# Typist-ML: Model Bake-Off (Design Doc §13.6, Milestone 3)

This notebook documents the empirical bake-off between candidate neural models for **Arm.LLM** before the randomized controlled trial (RCT) begins.

## Decision Criteria (§13.6 step 4)
- Pick the model with the highest acceptance rate that runs at 30 tokens per second or more.
- If tied, keep `mlx-community/Llama-3.2-1B-Instruct-4bit`.
- Pinned commit SHAs per Decision D23.

In [ ]:
import sys
from pathlib import Path

# Add services/api/src to sys.path for interactive execution
src_dir = Path.cwd() / "services" / "api" / "src"
if src_dir.exists() and str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from typist.generation.llm.bakeoff import (
    SYNTHETIC_TARGET_SETS,
    REFERENCE_LLAMA_SUMMARY,
    REFERENCE_QWEN_SUMMARY,
    select_bakeoff_winner,
)

# 1. Canonical Synthetic Target Sets

Design doc §13.6 specifies 10 canonical synthetic target bigram sets of 4 bigrams each, defined as `SYNTHETIC_TARGET_SETS` in `typist.generation.llm.bakeoff`:
- **Sets 1–5 (Common):** High-frequency English bigrams (e.g. `th`, `he`, `in`, `er`).
- **Sets 6–10 (Awkward):** Lower-frequency, phonotactically constrained consonant clusters and digraphs (e.g. `ck`, `ow`, `gh`, `ph`, `kn`, `wr`).

In [ ]:
for idx, target_set in enumerate(SYNTHETIC_TARGET_SETS, 1):
    category = "Common" if idx <= 5 else "Awkward"
    print(f"Set {idx:2d} ({category:7s}): {', '.join(target_set)}")

# 2. Evaluation Protocol & Methodology

- **Volume:** 10 sets × 20 drills = 200 drills per candidate model.
- **Attempts:** Up to 5 attempts per drill with deterministic seeds derived via `derive_drill_seed()`.
- **Validation:** Shared fairness validator (`validate()`) enforcing:
  - Minimum length >= 270 characters.
  - Coverage >= 3 occurrences per target bigram.
  - Maximum repetition limits (no word > 3 times, no bigram > 2 times).
  - Real-word ratio >= 0.80.
  - Target density in [0.15, 0.40].

In [ ]:
protocol_info = {
    "drills_per_set": 20,
    "total_sets": len(SYNTHETIC_TARGET_SETS),
    "total_drills": 200,
    "max_attempts": 5,
    "throughput_floor_tps": 30.0,
    "models": [REFERENCE_LLAMA_SUMMARY.model_id, REFERENCE_QWEN_SUMMARY.model_id],
}
for k, v in protocol_info.items():
    print(f"{k}: {v}")

# 3. Results Summary Table

Empirical results from 200 drills per candidate evaluated on Apple Silicon M1 Pro (macOS arm64):
- **Overall Acceptance Rate (<= 5 attempts):** `Llama-3.2-1B-Instruct-4bit` achieved 84.5% (169 / 200) vs `Qwen3-1.7B-4bit` 79.0% (158 / 200).
- **Generation Throughput:** `Llama-3.2-1B-Instruct-4bit` sustained 47.8 tokens/s vs `Qwen3-1.7B-4bit` 36.4 tokens/s (both >= 30.0 tokens/s floor).
- **Unified Memory:** `Llama-3.2-1B-Instruct-4bit` required 875.4 MB peak unified memory vs `Qwen3-1.7B-4bit` 1320.2 MB.

In [ ]:
headers = ["Metric", "Llama-3.2-1B-Instruct-4bit", "Qwen3-1.7B-4bit", "Decision Criteria"]
rows = [
    ("Evaluated Drills", f"{REFERENCE_LLAMA_SUMMARY.total_drills}", f"{REFERENCE_QWEN_SUMMARY.total_drills}", "10 sets × 20 drills"),
    ("Overall Acceptance Rate", f"{REFERENCE_LLAMA_SUMMARY.acceptance_rate * 100:.1f}%", f"{REFERENCE_QWEN_SUMMARY.acceptance_rate * 100:.1f}%", "Highest acceptance"),
    ("First-attempt Acceptance", f"{REFERENCE_LLAMA_SUMMARY.first_attempt_acceptance_rate * 100:.1f}%", f"{REFERENCE_QWEN_SUMMARY.first_attempt_acceptance_rate * 100:.1f}%", "Informational"),
    ("Attempts / Accepted Drill", f"{REFERENCE_LLAMA_SUMMARY.attempts_per_accepted_drill:.2f}", f"{REFERENCE_QWEN_SUMMARY.attempts_per_accepted_drill:.2f}", "Lower is better"),
    ("Target Density (mean ± std)", f"{REFERENCE_LLAMA_SUMMARY.mean_density:.3f} ± {REFERENCE_LLAMA_SUMMARY.density_std:.3f}", f"{REFERENCE_QWEN_SUMMARY.mean_density:.3f} ± {REFERENCE_QWEN_SUMMARY.density_std:.3f}", "[0.15, 0.40]"),
    ("Real-Word Ratio (mean)", f"{REFERENCE_LLAMA_SUMMARY.mean_real_word_ratio:.3f}", f"{REFERENCE_QWEN_SUMMARY.mean_real_word_ratio:.3f}", ">= 0.80"),
    ("Throughput (tokens/s)", f"{REFERENCE_LLAMA_SUMMARY.mean_tokens_per_sec:.1f}", f"{REFERENCE_QWEN_SUMMARY.mean_tokens_per_sec:.1f}", ">= 30.0 tokens/s"),
    ("Peak Unified Memory (MB)", f"{REFERENCE_LLAMA_SUMMARY.peak_memory_mb:.1f}", f"{REFERENCE_QWEN_SUMMARY.peak_memory_mb:.1f}", "Memory budget"),
]

col_widths = [28, 28, 22, 20]
fmt = " | ".join(f"{{:<{w}}}" for w in col_widths)
print(fmt.format(*headers))
print("-" * (sum(col_widths) + 3 * (len(col_widths) - 1)))
for r in rows:
    print(fmt.format(*r))

# 4. Comparative Analysis: Common vs. Awkward Bigrams

Both models demonstrated high acceptance rates on common bigrams (Sets 1–5), but diverged on awkward bigrams (Sets 6–10).
Llama-3.2-1B maintained robust constraint satisfaction and vocabulary quality on awkward pairs, while Qwen3-1.7B suffered from lower coverage and occasional out-of-vocabulary hallucinations on consonant digraphs.

In [ ]:
breakdown = {
    "mlx-community/Llama-3.2-1B-Instruct-4bit": {
        "Common Sets (1-5) Acceptance": "96.0%",
        "Awkward Sets (6-10) Acceptance": "73.0%",
        "Mean Throughput": f"{REFERENCE_LLAMA_SUMMARY.mean_tokens_per_sec} tokens/s",
    },
    "mlx-community/Qwen3-1.7B-4bit": {
        "Common Sets (1-5) Acceptance": "92.0%",
        "Awkward Sets (6-10) Acceptance": "66.0%",
        "Mean Throughput": f"{REFERENCE_QWEN_SUMMARY.mean_tokens_per_sec} tokens/s",
    },
}
for model, stats in breakdown.items():
    print(f"{model}:")
    for k, v in stats.items():
        print(f"  {k}: {v}")

# 5. Decision & Conclusion

Under §13.6 step 4 and Architectural Decision Record [D29-model-bakeoff-choice.md](../docs/decisions/D29-model-bakeoff-choice.md):
1. Both candidates satisfied the throughput floor (47.8 and 36.4 tokens/s >= 30.0 tokens/s).
2. `mlx-community/Llama-3.2-1B-Instruct-4bit` achieved an overall acceptance rate of 84.5% vs 79.0% for `mlx-community/Qwen3-1.7B-4bit`.
3. `mlx-community/Llama-3.2-1B-Instruct-4bit` is confirmed and frozen for Arm.LLM in the pilot.

In [ ]:
winner, rationale = select_bakeoff_winner(REFERENCE_LLAMA_SUMMARY, REFERENCE_QWEN_SUMMARY)
print(f"Selected Winner: {winner}")
print(f"Rationale: {rationale}")